# 1유형

In [18]:
# 1. 주어진 리스트에 대해 아래 과정을 차례로 수행한 최종 결괏값을 출력하시오.
# lst = [2,3,3.2,5,7.5,10,11.8,12,23,25,31.5,34]
# 제1사분위수와 제3사분위수를 구하시오.
# 제1사분위수와 제3사분위수 차이의 절댓값을 구하시오.
# 그 값의 소수점을 버린 후 정수로 출력하시오.
import numpy as np
lst = [2,3,3.2,5,7.5,10,11.8,12,23,25,31.5,34]
int(abs(np.quantile(lst, 0.25) - np.quantile(lst, 0.75)))

18

In [40]:
# 2. 주어진 facebook 데이터 세트는 페이스북 라이브에 대한 사용자 반응을 집계한 것이다.
# 이 중 love 반응(num_loves)와 wow 반응(num_wows)을 매우 긍정적인 반응이라고 정의할 때,
# 전체 반응 수(num_reaction)중 매우 긍정적인 반응 수가 차지하는 비율을 계산하시오.
# 그리고 그 비율이 0.5보다 작고 0.4보다 크며 유형이 비디오에 해당하는 건수를 정수로 출력하시오.
import pandas as pd
df = pd.read_csv('datasets/Part5/402_facebook.csv')
df['positive'] = df['num_loves'] + df['num_wows']
df['ratio'] = df['positive'] / df['num_reactions']
len(df.loc[(df['ratio'] > 0.4)&(df['ratio'] < 0.5)&(df['status_type']=='video')])

90

In [86]:
# 3. 주어진 netflix 데이터 세트는 넷플릭스에 등록된 컨텐츠의 메타 데이터이다.
# 2018년 1월에 넷플릭스에 등록된 컨텐츠 중에서 'United Kingdom'이 단독 제작한 컨텐츠의 수를 정수로 출력하시오.
df = pd.read_csv('datasets/Part5/403_netflix.csv')
df['date_added'] = df['date_added'].astype('datetime64[ns]')
target = df.loc[(df['date_added'].dt.to_period('M')=='2018-01')&(df['country']=='United Kingdom')]
len(target)

6

# 2유형

In [ ]:
# 1. 다음은 Customer Segmentation 데이터 세트이다.
# 주어진 훈련 데이터 세트를 활용하여 고객이 속한 세그먼트(Segmentation)를 예측하고 해당 예측 결과를 csv 파일로 제출하시오.
# 결과 제출 양식 : 제출한 예측값의 macro_f1 결과를 통해 영역별 배점에 따라 최종 점수가 반영될 예정
# | ID | Segmentation |
# |----|-------|
# | 1  | A |
# | 2  | A |
# | 3  | C |
# | ... | ... |
# [결과 제출 양식]

# | 변수 | 설명 |
# |-----|------|
# | ID | 고객 ID 번호 |
# | Gender | 성별 |
# | Ever_Married | 결혼 여부 |
# | Age | 나이 |
# | Graduated | 대학 졸업 여부 |
# | Profession | 직업 |
# | Work_Experience | 근무 연수 |
# | Spending_Score | 지출 수준 |
# | Family_Size | 가족 수(본인 포함) |
# | Segmentation | 고객 세그먼트(A,B,C,D 중 하나) |
# [Customer Segmentation 데이터 세트 변수 설명]

In [235]:
import pandas as pd
X_train = pd.read_csv('datasets/Part5/404_x_train.csv')
y_train = pd.read_csv('datasets/Part5/404_y_train.csv')
X_test = pd.read_csv('datasets/Part5/404_x_test.csv')

# X_train.info(), y_train.info(), X_test.info()
# print(X_train.shape, y_train.shape, X_test.shape)

X_full = pd.concat([X_train, X_test])
X_full = pd.get_dummies(X_full)
X_train = X_full[:X_train.shape[0]]
X_test = X_full[X_train.shape[0]:]

X_test_id = X_test.pop('ID')
X_train = X_train.drop(columns=['ID'])
y = y_train['Segmentation']

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
# X_train.shape, X_val.shape, y_train.shape, y_val.shape

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_val_pred = model.predict(X_val)

from sklearn.metrics import f1_score, accuracy_score
f1 = f1_score(y_val, y_val_pred, average='macro')
acc = accuracy_score(y_val, y_val_pred)
print(f1, acc)

pred = model.predict(X_test)
result = pd.DataFrame({'ID':X_test_id, 'Segmentation':pred})
result.to_csv('result.csv', index=False)

0.46615311336074305 0.47023809523809523


In [303]:
import pandas as pd
X_train = pd.read_csv('datasets/Part5/404_x_train.csv')
y_train = pd.read_csv('datasets/Part5/404_y_train.csv')
X_test = pd.read_csv('datasets/Part5/404_x_test.csv')

COL_DEL = ['ID']
COL_NUM = ['Age','Work_Experience','Family_Size']
COL_CAT = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score']
COL_Y = ['Segmentation']

from sklearn.preprocessing import LabelEncoder
le_y = LabelEncoder()
y_train['Segmentation'] = le_y.fit_transform(y_train['Segmentation'])

from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train[COL_NUM+COL_CAT], y_train[COL_Y].values.ravel(),\
                                            test_size=0.3, stratify=y_train[COL_Y])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_tr[COL_NUM])
X_tr[COL_NUM] = scaler.transform(X_tr[COL_NUM])
X_val[COL_NUM] = scaler.transform(X_val[COL_NUM])
X_test[COL_NUM] = scaler.transform(X_test[COL_NUM])

from sklearn.preprocessing import LabelEncoder
X = pd.concat([X_train[COL_CAT], X_test[COL_CAT]])

for col in COL_CAT:
    le = LabelEncoder()
    le.fit(X[col])
    X_tr[col] = le.transform(X_tr[col])
    X_val[col] = le.transform(X_val[col])
    X_test[col] = le.transform(X_test[col])

from sklearn.ensemble import RandomForestClassifier
modelRF = RandomForestClassifier(random_state=123)
modelRF.fit(X_tr, y_tr)

from xgboost import XGBClassifier
modelXGB = XGBClassifier(random_state=123)
modelXGB.fit(X_tr, y_tr)

y_val_predRF = modelRF.predict(X_val)
y_val_predXGB = modelXGB.predict(X_val)

from sklearn.metrics import f1_score, accuracy_score
scoreRF = f1_score(y_val, y_val_predRF, average='macro')
scoreXGB = f1_score(y_val, y_val_predXGB, average='macro')
print(scoreRF, scoreXGB)

accRF = accuracy_score(y_val, y_val_predRF)
accXGB = accuracy_score(y_val, y_val_predXGB)
print(accRF, accXGB)

y_tr_predRF = modelRF.predict(X_tr)
y_tr_predXGB = modelXGB.predict(X_tr)
scoreRF = f1_score(y_tr, y_tr_predRF, average='macro')
scoreXGB = f1_score(y_tr, y_tr_predXGB, average='macro')
print(scoreRF, scoreXGB)

pred = modelXGB.predict(X_test[COL_NUM+COL_CAT])
pred = le_y.inverse_transform(pred)

result = pd.DataFrame({'ID':X_test['ID'], 'Segmentation':pred})
result.to_csv('result.csv', index=False)

0.4752760712246328 0.48660277007748787
0.48412698412698413 0.49503968253968256
0.9454921789761076 0.7939740288718997
